# 4.0 Evaluacion y Resultados

Corresponde a la seccion 7.5 de la monografia. Punto central: el modelo
seleccionado en 3.0 (sobre el conjunto de ajuste) se reentrena con TODO
el conjunto de ajuste y se evalua **una unica vez** sobre el conjunto de
test independiente, que no participo en ninguna comparacion previa.

Este es el resultado que responde directamente a la observacion del
docente: *"las mismas cuatro particiones se utilizan para comparar
modelos, seleccionar Random Forest y reportar el desempeno final. No
existe un test independiente posterior a la seleccion."*

In [1]:
import sys
sys.path.insert(0, "..")

import joblib
import pandas as pd
from rodamientos_module import config
from rodamientos_module.modeling import train as train_module
from rodamientos_module.modeling.models import (
    BaselineModel, LogisticRegressionModel, SVMModel, RandomForestModel,
)

df_ajuste = pd.read_csv(config.DIR_DATOS_PROCESSED / "conjunto_ajuste.csv")
df_test_final = pd.read_csv(config.DIR_DATOS_PROCESSED / "conjunto_test_final.csv")

seleccion = joblib.load(config.DIR_MODELOS / "seleccion_hiperparametros.pkl")
mejor_rf = seleccion["mejor_rf"]
mejor_svm = seleccion["mejor_svm"]
modelo_seleccionado_nombre = seleccion["modelo_seleccionado"]

print(f"Modelo seleccionado en 3.0: {modelo_seleccionado_nombre}")
print(f"Hiperparametros RF: {mejor_rf}")
print(f"Hiperparametros SVM: {mejor_svm}")

Modelo seleccionado en 3.0: SVM (ajustado)
Hiperparametros RF: {'n_estimators': 50, 'max_depth': None}
Hiperparametros SVM: {'kernel': 'rbf', 'C': np.float64(0.1)}


## Evaluacion final sobre el test independiente

Se reconstruye el modelo seleccionado con sus hiperparametros ya
ajustados, se reentrena con los 18 archivos de ajuste, y se evalua una
unica vez sobre los 6 archivos de test.

In [2]:
mapa_modelos = {
    "Baseline": BaselineModel(),
    "Regresion Logistica": LogisticRegressionModel(random_state=config.RANDOM_STATE),
    "SVM (ajustado)": SVMModel(random_state=config.RANDOM_STATE, **mejor_svm),
    "Random Forest (ajustado)": RandomForestModel(random_state=config.RANDOM_STATE, **mejor_rf),
}

modelo_final_base = mapa_modelos[modelo_seleccionado_nombre]

resultado_final, matriz_final, modelo_final_entrenado = train_module.evaluar_modelo_final(
    df_ajuste, df_test_final, modelo_final_base
)

print(f"=== Desempeno final ({modelo_seleccionado_nombre}) sobre test independiente ===")
for metrica, valor in resultado_final.items():
    if metrica != "modelo":
        print(f"{metrica}: {valor:.4f}")

=== Desempeno final (SVM (ajustado)) sobre test independiente ===
accuracy: 1.0000
balanced_accuracy: 1.0000
f1_macro: 1.0000


### Matriz de confusion sobre el test independiente

Orden de clases: Normal, Ball, Inner Race, Outer Race.

In [3]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(matriz_final)
ax.set_xticks(range(len(config.ORDEN_CLASES)))
ax.set_yticks(range(len(config.ORDEN_CLASES)))
ax.set_xticklabels(config.ORDEN_CLASES, rotation=45, ha="right")
ax.set_yticklabels(config.ORDEN_CLASES)
ax.set_xlabel("Condicion predicha")
ax.set_ylabel("Condicion real")
ax.set_title(f"Matriz de confusion - Test independiente ({modelo_seleccionado_nombre})")
for i in range(len(config.ORDEN_CLASES)):
    for j in range(len(config.ORDEN_CLASES)):
        ax.text(j, i, matriz_final[i, j], ha="center", va="center")
plt.tight_layout()
plt.savefig("../reports/figures/fig_matriz_confusion_test_final.png", dpi=150)
plt.show()

**Nota para el informe (reemplaza el numero final de la Tabla 12/13 y la
seccion 7.5.6):**

> *"El modelo [MODELO], con los hiperparametros [...] seleccionados
> mediante comparacion sobre el conjunto de ajuste, fue reentrenado con
> los 18 archivos de ajuste y evaluado una unica vez sobre los 6
> archivos reservados como test independiente, obteniendo una exactitud
> balanceada de [Z] y un F1 macro de [Z]. A diferencia de la validacion
> cruzada usada para la seleccion, este resultado no participo en
> ninguna decision de modelado, por lo que constituye una estimacion mas
> realista del desempeno esperado sobre registros no vistos durante el
> proceso de seleccion."*

Completar `[MODELO]` y `[Z]` con los valores reales impresos arriba.

## Importancia de caracteristicas (importancia por permutacion)

Se calcula sobre el modelo final ya entrenado (independientemente de
cual haya sido seleccionado), evaluando el descenso de balanced
accuracy sobre el conjunto de test independiente al permutar cada
caracteristica. En la version original de la monografia este analisis
se documentaba solo para Random Forest (seccion 7.5.3); como el
modelo finalmente seleccionado es SVM, se recalcula aqui sobre SVM.


In [4]:
from sklearn.inspection import permutation_importance

importancia = permutation_importance(
    modelo_final_entrenado, df_test_final[config.CARACTERISTICAS_MODELO],
    df_test_final[config.COLUMNA_CLASE],
    scoring="balanced_accuracy", n_repeats=10,
    random_state=config.RANDOM_STATE, n_jobs=-1,
)

tabla_importancia = pd.DataFrame({
    "caracteristica": config.CARACTERISTICAS_MODELO,
    "importancia_media": importancia.importances_mean,
    "desviacion": importancia.importances_std,
}).sort_values("importancia_media", ascending=False).reset_index(drop=True)

tabla_importancia.to_csv("../reports/tabla_importancia_test_final.csv", index=False)
print(f"Importancia por permutacion calculada sobre {modelo_seleccionado_nombre}")
tabla_importancia.round(4)


Importancia por permutacion calculada sobre SVM (ajustado)


       caracteristica  importancia_media  desviacion
0  centroide_espectral           0.245665    0.025758
1                   rms           0.224138    0.019751
2              kurtosis           0.140421    0.024979
3              skewness           0.064080    0.006604
4         factor_cresta           0.049425    0.016112

## Guardado del modelo final

Se guarda el modelo ya entrenado con todo el conjunto de ajuste, listo
para usarse con `rodamientos_module.modeling.predict.predecir_senal`
(seccion 7.6, propuesta de utilizacion).

In [5]:
joblib.dump(modelo_final_entrenado, config.DIR_MODELOS / "modelo_final.pkl")
print(f"Modelo guardado en models/modelo_final.pkl ({modelo_seleccionado_nombre})")

Modelo guardado en models/modelo_final.pkl (SVM (ajustado))
